# Memory-Based Collaborative Filtering

Evaluate User-KNN and Item-KNN across:
- Similarity measures: cosine vs. Pearson / adjusted-cosine
- Neighbourhood sizes: K = 10, 20, 40
- Metrics: RMSE, MAE, Precision@10, Recall@10, NDCG@10

In [ ]:
import sys, os

# Works both locally (run from notebooks/) and on Google Colab (run from repo root)
_src = os.path.join('..', 'src') if os.path.isdir(os.path.join('..', 'src')) else 'src'
_data = os.path.join('..', 'data', 'processed') if os.path.isdir(os.path.join('..', 'data')) else os.path.join('data', 'processed')
sys.path.insert(0, _src)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import get_data
from memory_based.user_based_cf import UserBasedCF
from memory_based.item_based_cf import ItemBasedCF
from evaluation.metrics import evaluate_predictions, evaluate_ranking, catalog_coverage

sns.set_theme(style='whitegrid')
%matplotlib inline

In [2]:
train, test, matrix, movies = get_data()
print(f'Train: {len(train):,}  Test: {len(test):,}  Matrix: {matrix.shape}')

Train: 79,619  Test: 20,381  Matrix: (943, 1650)


## Helper: build test-relevant sets

A test item is *relevant* if the user rated it ≥ 4 stars.

In [3]:
RELEVANCE_THRESHOLD = 4
K = 10
N_ITEMS = matrix.shape[1]
ALL_ITEMS = list(matrix.columns)

test_relevant = (
    test[test['rating'] >= RELEVANCE_THRESHOLD]
    .groupby('user_id')['item_id']
    .apply(set)
    .to_dict()
)

def get_rated(user_id, train_df):
    return set(train_df[train_df['user_id'] == user_id]['item_id'])

## 1. User-Based CF

In [ ]:
results = []

for sim in ['cosine', 'pearson']:
    for k in [10, 20, 40]:
        print(f'User-KNN  sim={sim}  k={k} ...', end=' ')
        model = UserBasedCF(k=k, similarity=sim).fit(matrix)

        # Rating prediction
        test['predicted'] = model.predict_batch(test)
        pred_metrics = evaluate_predictions(test, pred_col='predicted')

        # Top-N ranking
        test_users = test['user_id'].unique()
        recs = {}
        for uid in test_users:
            rated = get_rated(uid, train)
            recs[uid] = model.recommend(uid, n=K)

        rank_metrics = evaluate_ranking(recs, test_relevant, k=K)
        cov = catalog_coverage(list(recs.values()), N_ITEMS)

        results.append({'Model': 'User-KNN', 'Similarity': sim, 'K': k,
                        **pred_metrics, **rank_metrics, 'Coverage': cov})
        print(f"RMSE={pred_metrics['RMSE']:.4f}  NDCG@10={rank_metrics['NDCG@10']:.4f}")

results_df = pd.DataFrame(results)
results_df

## 2. Item-Based CF

In [ ]:
item_results = []

for sim in ['cosine', 'adjusted_cosine']:
    for k in [10, 20, 40]:
        print(f'Item-KNN  sim={sim}  k={k} ...', end=' ')
        model = ItemBasedCF(k=k, similarity=sim).fit(matrix)

        test['predicted'] = model.predict_batch(test)
        pred_metrics = evaluate_predictions(test, pred_col='predicted')

        recs = {}
        for uid in test['user_id'].unique():
            recs[uid] = model.recommend(uid, n=K)

        rank_metrics = evaluate_ranking(recs, test_relevant, k=K)
        cov = catalog_coverage(list(recs.values()), N_ITEMS)

        item_results.append({'Model': 'Item-KNN', 'Similarity': sim, 'K': k,
                              **pred_metrics, **rank_metrics, 'Coverage': cov})
        print(f"RMSE={pred_metrics['RMSE']:.4f}  NDCG@10={rank_metrics['NDCG@10']:.4f}")

item_results_df = pd.DataFrame(item_results)
item_results_df

## 3. Best configurations — side-by-side

In [ ]:
all_results = pd.concat([results_df, item_results_df], ignore_index=True)
all_results.sort_values('RMSE').to_csv(os.path.join(_data, 'memory_based_results.csv'), index=False)

# Best per model type
best = all_results.loc[all_results.groupby('Model')['RMSE'].idxmin()]
best[['Model', 'Similarity', 'K', 'RMSE', 'MAE', 'Precision@10', 'Recall@10', 'NDCG@10', 'Coverage']]

## 4. Effect of K on RMSE

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, model_name, res in zip(axes, ['User-KNN', 'Item-KNN'], [results_df, item_results_df]):
    for sim, grp in res.groupby('Similarity'):
        ax.plot(grp['K'], grp['RMSE'], marker='o', label=sim)
    ax.set_title(f'{model_name} — RMSE vs K')
    ax.set_xlabel('K (neighbours)')
    ax.set_ylabel('RMSE')
    ax.legend()

plt.tight_layout()
plt.show()

## 5. Sample recommendations for a user

In [ ]:
SAMPLE_USER = 1

best_user_model = UserBasedCF(k=20, similarity='pearson').fit(matrix)
top10 = best_user_model.recommend(SAMPLE_USER, n=10)

print(f'Top-10 recommendations for user {SAMPLE_USER} (User-KNN, Pearson, K=20):')
movies[movies['item_id'].isin(top10)][['item_id', 'title']]